# Bagging and Restart-Ensemble Experiment

## Why add ensembles?

The six-supergroup retrieval experiment compared one selected PLSA fit with full-data TF-IDF and LSI. In that run, PLSA's single best-restart model performed poorly. We then tested aggregation because PLSA uses stochastic EM and different fits can produce different document rankings. The preliminary PLSA-only ensemble looked promising, but it was not a fair ensemble comparison while TF-IDF and LSI were represented by only one model each.

This experiment applies five-member bootstrap ensembles to PLSA, TF-IDF, and LSI. Each member is trained from a same-size sample drawn with replacement from the entire training partition, and all three methods use the same five saved resamples. We combine rankings by mean rank because raw scores from the three model families are on different scales. Alongside the full-data references, PLSA also has a five-restart ensemble trained on the complete training data; that control helps show what aggregation across random initializations can achieve without bootstrap resampling.

The retrieval task remains the same six human-defined supergroups, one fixed query per group, and the same held-out candidates. This isolates the ensemble question within the task already established in `Six_Category_Experiment.ipynb`.


All shared implementations are imported from `src/`. Run this notebook from the repository root. Old execution outputs were cleared during refactoring. The final cell writes tables and metadata for `Result.py`; experiment-specific loops remain here.

## Environment

The next cell contains the optional dependency-install command for notebook environments where these packages are not already available.


In [ ]:
# Install from the repository root: pip install -r requirements.txt
# Then: python -m spacy download en_core_web_sm


## 1. Libraries and shared settings

The imports cover sparse count matrices and PLSA, TF-IDF and LSI, the shared split and supergroup mapping, retrieval scoring, and result tables and plots.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42


from src.config import EXPERIMENTS, SUPERGROUPS, CATEGORY_QUERIES, SUPERGROUP_QUERIES, QUERY_BANK
from src.dataset import load_corpus
from src.preprocessing import preprocess_texts, split_train_test, split_train_val_test, vectorize_counts
from src.plsa_model import PLSA, train_plsa_restarts
from src.retrieval import fit_baselines, plsa_query_log_likelihood, background_word_probabilities
from src.metrics import metrics_at_k
from src.experiments import make_mixed_document, average_rank_score
from src.results import export_results
EXPERIMENT = 'Bagging_Experiment'
CONFIG = EXPERIMENTS[EXPERIMENT]


## 2. Reuse the six-supergroup evaluation design

The same mapping of all 20 categories into six topical groups is declared here so this notebook can run independently. It defines stratification and relevance; the labels are not inputs to PLSA training. Keeping the mapping, split, text processing, queries, and held-out candidates aligned with the preceding experiment makes the ensemble comparison address model aggregation rather than a different retrieval task.


In [ ]:
# Shared loader; corpus selection remains an explicit experimental choice.
corpus = load_corpus(CONFIG["corpus"], seed=SEED)
documents, labels, target_names = corpus.documents, corpus.labels, corpus.target_names
original_labels = corpus.original_labels
source_category_names = corpus.source_category_names
source_to_supergroup = {category: group for group, members in SUPERGROUPS.items() for category in members}
print("Corpus:", CONFIG["corpus"], "Documents:", len(documents))
for i, label in enumerate(target_names):
    print(label, int((labels == i).sum()))


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Corpus and group counts

The output confirms that all 20 categories are assigned once and that the full 18,846-document corpus is used. Group sizes remain naturally different; no training examples are discarded to equalize them. This is the same relevance taxonomy used in the six-supergroup retrieval experiment.


## 3. Shared split and preprocessing

Use a stratified 80/20 split and fit the 3,000-term vocabulary on training text only. The bootstrap samples will be drawn from the complete training partition. Every method will rank the same non-empty test documents.


In [ ]:
# Split raw text first. Fit vocabulary only on training text.
docs_train, docs_test, y_train, y_test, original_y_train, original_y_test = split_train_test(documents, labels, original_labels, test_size=CONFIG["test_size"], seed=SEED)
docs_train_processed = preprocess_texts(docs_train)
docs_test_processed = preprocess_texts(docs_test)
count_vectorizer, X_train, X_test = vectorize_counts(docs_train_processed, docs_test_processed)
vocabulary = count_vectorizer.get_feature_names_out()
nonempty_test_mask = np.asarray([bool(doc.strip()) for doc in docs_test_processed])
print("Train/test count matrices:", X_train.shape, X_test.shape)
print("Non-empty test texts:", int(nonempty_test_mask.sum()))


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Split and text representation

The split retains 15,076 training and 3,770 held-out documents. The shared count vocabulary has 3,000 terms; documents emptied by preprocessing have no usable vector and are excluded from retrieval rankings. The labels affect stratification and later scoring only.

## 4. PLSA representation

PLSA represents each document as a mixture over latent topics and each topic as a distribution over words. EM estimates these distributions from word counts. The class below uses sparse input matrices; it does not use Newsgroups labels in its updates.


In [ ]:
# The single EM and fold-in implementation lives in src/plsa_model.py.
from src.plsa_model import PLSA


## 5. Full-data PLSA reference and restart control

First fit five PLSA initializations to the complete training matrix using K=6 and the same settings as the preceding experiment. The highest-training-likelihood run is the single-model reference. The five full-data runs are also folded into the held-out documents and retained for a mean-rank ensemble. This restart control combines different initializations without resampling documents.


In [ ]:
# train_plsa_restarts is imported from src above.

K = 6
best_plsa, restart_runs = train_plsa_restarts(
    X_train, n_topics=K, n_restarts=5, max_iter=100
)
theta_train = best_plsa.transform()
theta_test = best_plsa.fold_in(X_test, max_iter=100)
print("theta_train:", theta_train.shape)
print("theta_test:", theta_test.shape)


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Full-data PLSA output

The five full-data restarts provide both the single-model reference (the run with the highest training likelihood) and the models for a restart ensemble. The restart ensemble changes the number of rankings combined while keeping the training corpus unchanged.

## 6. Train the PLSA bootstrap members

Each of five members draws 15,076 row indices with replacement from the complete training matrix and fits a fresh K=6 PLSA model. A member therefore repeats some documents and omits others, while every training document remains eligible for selection. The index arrays are saved for reuse by TF-IDF and LSI.

In [ ]:
BAGGING_ESTIMATORS = 5
bootstrap_indices_list = []
bagged_plsa_models = []
bootstrap_rng = np.random.default_rng(SEED + 500)
for estimator_id in range(BAGGING_ESTIMATORS):
    bootstrap_indices = bootstrap_rng.integers(0, X_train.shape[0], size=X_train.shape[0])
    bootstrap_indices_list.append(bootstrap_indices)
    bootstrap_X = X_train[bootstrap_indices]
    model = PLSA(
        n_topics=K, max_iter=100, tol=1e-4,
        random_state=SEED + 1000 + estimator_id, beta=1.0
    ).fit(bootstrap_X)
    bagged_plsa_models.append(model)
    print(
        f"Bootstrap model {estimator_id + 1}/{BAGGING_ESTIMATORS}: "
        f"{len(np.unique(bootstrap_indices))} unique training documents, "
        f"final bootstrap log-likelihood={model.log_likelihoods[-1]:.3f}"
    )

bagged_theta_test = [model.fold_in(X_test, max_iter=100) for model in bagged_plsa_models]
restart_theta_test = [model.fold_in(X_test, max_iter=100) for model, _ in restart_runs]
print("Bagged PLSA models:", len(bagged_plsa_models))
print("The saved bootstrap indices will be reused for TF-IDF and LSI.")


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Bootstrap PLSA output

Each 15,076-row bootstrap sample contains about 9,477–9,605 unique training documents; remaining rows are repeats. The five members therefore see different empirical samples. Their rankings will be combined with mean rank, and their exact indices are shared with the TF-IDF and LSI bootstrap members.

## 7. Full-training TF-IDF and LSI references

Fit each baseline once on the complete 80% training partition. These reference models are compared with five-member bootstrap ensembles fitted using the shared resample indices. The vocabulary and all baseline models are learned from training text only.


In [ ]:
# All baselines use the shared train-only fitting implementation.
tfidf_vectorizer, lsi_model, tfidf_splits, lsi_splits = fit_baselines(
    docs_train_processed, docs_test_processed, seed=SEED
)
X_train_tfidf, X_test_tfidf = tfidf_splits
lsi_train, lsi_test = lsi_splits



> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Full-data baseline output

TF-IDF uses the training-only 3,000-term representation, and LSI projects it into 50 dimensions. These full-training models are the non-ensemble references for their method families.

## 8. Fit bootstrap TF-IDF and LSI members

For each saved resample, fit a fresh TF-IDF vocabulary and IDF weights, then fit a fresh 50-dimensional LSI space in that member's vocabulary. Every member transforms the same held-out documents. Re-fitting within each sample is part of the bootstrap procedure; sharing the indices pairs the document draws across model families.

In [ ]:
bagged_tfidf_vectorizers = []
bagged_tfidf_test = []
bagged_lsi_models = []
bagged_lsi_test = []

for member_id, bootstrap_indices in enumerate(bootstrap_indices_list):
    bootstrap_docs = [docs_train_processed[i] for i in bootstrap_indices]
    member_vectorizer, member_lsi, member_tfidf, member_lsi_arrays = fit_baselines(
        bootstrap_docs, docs_test_processed, seed=SEED + 1200 + member_id
    )
    member_train_tfidf, member_test_tfidf = member_tfidf
    _, member_test_lsi = member_lsi_arrays
    member_components = member_test_lsi.shape[1]
    bagged_tfidf_vectorizers.append(member_vectorizer)
    bagged_tfidf_test.append(member_test_tfidf)
    bagged_lsi_models.append(member_lsi)
    bagged_lsi_test.append(member_test_lsi)
    print(
        f"Baseline bootstrap {member_id + 1}/{BAGGING_ESTIMATORS}: "
        f"TF-IDF shape={member_train_tfidf.shape}, LSI dimensions={member_components}"
    )


### Bootstrap baseline output

Five TF-IDF and five LSI members were fitted from the same resampled document indices as the PLSA members. Each model family keeps its own fitted vocabulary and latent space, while all members rank the same held-out documents.


## 9. Retrieval metrics and queries

Each of the six broad supergroups has one fixed query, and relevance means membership in that supergroup. All methods rank the same non-empty held-out candidates. The table reports every query and the macro-average, giving each query equal weight.

Precision@10 and Recall@10 use the top ten. AP@10 averages precision at relevant ranks through rank 10, normalized by the smaller of ten and the number of relevant held-out documents; MAP@10 is the mean over six queries. NDCG@10 uses binary relevance and rank discounts. These category labels are proxy relevance judgments.


In [ ]:
supergroup_queries = SUPERGROUP_QUERIES

missing = sorted(set(target_names) - set(supergroup_queries))
extra = sorted(set(supergroup_queries) - set(target_names))
print("Supergroups without a query:", missing)
print("Query names not found in supergroups:", extra)
if missing or extra:
    raise ValueError("Provide exactly one query for each evaluation supergroup.")

nonempty_test_mask = np.asarray([bool(doc.strip()) for doc in docs_test_processed])
print("Non-empty test documents:", int(nonempty_test_mask.sum()), "/", len(docs_test_processed))

# plsa_query_log_likelihood is imported from src above.


# average_rank_score is imported from src above.

def evaluate_query(query_text, target_label, k=10):
    processed_query = preprocess_texts([query_text])[0]
    q_tfidf = tfidf_vectorizer.transform([processed_query])
    q_lsi = lsi_model.transform(q_tfidf)
    q_counts = count_vectorizer.transform([processed_query]).astype(np.float64)
    q_plsa = best_plsa.fold_in(q_counts, max_iter=100)
    y_eval = y_test[nonempty_test_mask]

    tfidf_scores = cosine_similarity(q_tfidf, X_test_tfidf).ravel()[nonempty_test_mask]
    lsi_scores = cosine_similarity(q_lsi, lsi_test).ravel()[nonempty_test_mask]
    plsa_cosine_scores = cosine_similarity(q_plsa, theta_test).ravel()[nonempty_test_mask]
    plsa_likelihood_scores = plsa_query_log_likelihood(q_counts, theta_test, best_plsa.phi_)[nonempty_test_mask]

    restart_cosine_scores = []
    restart_likelihood_scores = []
    for (model, _), member_theta_test in zip(restart_runs, restart_theta_test):
        member_query_theta = model.fold_in(q_counts, max_iter=100)
        restart_cosine_scores.append(cosine_similarity(member_query_theta, member_theta_test).ravel()[nonempty_test_mask])
        restart_likelihood_scores.append(plsa_query_log_likelihood(q_counts, member_theta_test, model.phi_)[nonempty_test_mask])

    bootstrap_plsa_cosine_scores = []
    bootstrap_plsa_likelihood_scores = []
    for model, member_theta_test in zip(bagged_plsa_models, bagged_theta_test):
        member_query_theta = model.fold_in(q_counts, max_iter=100)
        bootstrap_plsa_cosine_scores.append(cosine_similarity(member_query_theta, member_theta_test).ravel()[nonempty_test_mask])
        bootstrap_plsa_likelihood_scores.append(plsa_query_log_likelihood(q_counts, member_theta_test, model.phi_)[nonempty_test_mask])

    bootstrap_tfidf_scores = []
    bootstrap_lsi_scores = []
    for vectorizer, member_test_tfidf, model, member_test_lsi in zip(
        bagged_tfidf_vectorizers, bagged_tfidf_test, bagged_lsi_models, bagged_lsi_test
    ):
        member_query_tfidf = vectorizer.transform([processed_query])
        bootstrap_tfidf_scores.append(cosine_similarity(member_query_tfidf, member_test_tfidf).ravel()[nonempty_test_mask])
        member_query_lsi = model.transform(member_query_tfidf)
        bootstrap_lsi_scores.append(cosine_similarity(member_query_lsi, member_test_lsi).ravel()[nonempty_test_mask])

    methods = {
        "TF-IDF/VSM (full)": tfidf_scores,
        "TF-IDF/VSM (bagged)": average_rank_score(bootstrap_tfidf_scores),
        "LSI (full)": lsi_scores,
        "LSI (bagged)": average_rank_score(bootstrap_lsi_scores),
        "PLSA cosine (best restart)": plsa_cosine_scores,
        "PLSA cosine (5 restarts)": average_rank_score(restart_cosine_scores),
        "PLSA cosine (bagged)": average_rank_score(bootstrap_plsa_cosine_scores),
        "PLSA query likelihood (best restart)": plsa_likelihood_scores,
        "PLSA query likelihood (5 restarts)": average_rank_score(restart_likelihood_scores),
        "PLSA query likelihood (bagged)": average_rank_score(bootstrap_plsa_likelihood_scores),
    }

    rows = []
    total_relevant = int((y_eval == target_label).sum())
    if total_relevant == 0:
        return rows

    for method_name, scores in methods.items():
        metric = metrics_at_k(scores, y_eval, target_label, k=k, suffix="k")
        rows.append({
            "query_supergroup": target_names[target_label],
            "method": method_name,
            "Precision@10": metric["Precision@k"],
            "Recall@10": metric["Recall@k"],
            "MAP@10": metric["AP@k"],
            "NDCG@10": metric["NDCG@k"],
            "relevant_test_documents": total_relevant,
        })
    return rows

eval_rows = []
for category_name in target_names:
    eval_rows.extend(evaluate_query(supergroup_queries[category_name], target_names.index(category_name), k=10))

evaluation_df = pd.DataFrame(eval_rows)
metric_columns = ["Precision@10", "Recall@10", "MAP@10", "NDCG@10"]
print("Per-query results:")
display(evaluation_df[["query_supergroup", "method", *metric_columns, "relevant_test_documents"]].round(4))
mean_metrics = (
    evaluation_df.groupby("method", sort=False)[metric_columns]
    .mean().reset_index().rename(columns={"method": "Method"})
)
print("Macro-average across the six queries:")
display(mean_metrics.round(4))


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### What the macro-average table shows

For PLSA cosine, Precision@10/MAP@10/NDCG@10 increase from 0.4333/0.2858/0.4248 for the best restart to 0.9167/0.8845/0.9356 for the five-restart ensemble and 0.9500/0.9196/0.9505 for the bootstrap ensemble. Query-likelihood PLSA rises from 0.4833/0.3567/0.4892 to 0.9000/0.8260/0.8937 with full-data restarts and 0.9333/0.9138/0.9491 with bootstrapping.

TF-IDF bagging changes MAP@10/NDCG@10 only slightly (0.8189/0.8982 to 0.8220/0.8990), while LSI bagging is slightly lower than its full-data reference (0.7155/0.8077 to 0.6869/0.7988). In this run, the PLSA five-restart control already exceeds the TF-IDF full-data reference on the macro metrics; bootstrap aggregation raises the PLSA averages again. This indicates that combining PLSA rankings is especially effective in this evaluation, and that the gain is not attributable to resampling alone.

The per-query table shows how the six groups contribute to these averages. Recall@10 is small because each query has hundreds of relevant documents and retrieves only ten. The plots present one metric at a time so each scale remains readable.

In [ ]:
metric_columns = ["Precision@10", "Recall@10", "MAP@10", "NDCG@10"]
method_colors = dict(zip(mean_metrics["Method"], plt.cm.tab10.colors))
method_labels = {
    "TF-IDF/VSM (full)": "TF-IDF\n(full)",
    "TF-IDF/VSM (bagged)": "TF-IDF\n(bagged)",
    "LSI (full)": "LSI\n(full)",
    "LSI (bagged)": "LSI\n(bagged)",
    "PLSA cosine (best restart)": "PLSA cosine\n(best restart)",
    "PLSA cosine (5 restarts)": "PLSA cosine\n(5 full-data restarts)",
    "PLSA cosine (bagged)": "PLSA cosine\n(bagged)",
    "PLSA query likelihood (best restart)": "PLSA query likelihood\n(best restart)",
    "PLSA query likelihood (5 restarts)": "PLSA query likelihood\n(5 full-data restarts)",
    "PLSA query likelihood (bagged)": "PLSA query likelihood\n(bagged)",
}

for metric in metric_columns:
    values = mean_metrics[metric].to_numpy(dtype=float)
    upper_limit = min(1.15, max(0.05, float(values.max()) * 1.22))
    plot_labels = [method_labels[method] for method in mean_metrics["Method"]]
    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.bar(plot_labels, values, color=[method_colors[m] for m in mean_metrics["Method"]])
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_ylim(0, upper_limit)
    ax.set_title(f"Macro-average {metric} by method")
    ax.set_ylabel(metric)
    ax.set_xlabel("Retrieval method")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Metric plot interpretation

These four charts display the macro-average results summarized above, with one metric per chart and method names on the horizontal axis. They make the large PLSA improvement from aggregation visible without forcing Recall@10 and the other metrics onto a shared scale. Read the per-query and macro-average tables for exact values.

## Export this run

Save result tables, convergence, topic words, settings and source hashes. `Result.py` reads these files without repeating model training.

In [ ]:
# This is reached only after all experiment cells complete successfully.
result_directory = export_results(EXPERIMENT, globals())
print("Results saved to:", result_directory)
